In [0]:
"""Silver: dim_anlage — unified renewable installation dimension.

UNION ALL across the four technology bronze tables, projecting to a
common 16-column schema. Filters out rows with invalid PLZ or
missing Bundesland (those can't contribute to geographic analysis).

Decommissioned installations are KEPT (they received historical EEG
payments). Filter on betriebs_status downstream when needed.
"""
from pyspark.sql import DataFrame
from pyspark.sql import functions as F


def select_common(table: str, technologie: str) -> DataFrame:
    return (
        spark.table(f"eeg_dev.bronze.mastr_{table}")
        .select(
            F.col("EinheitMastrNummer").cast("string").alias("mastr_nummer"),
            F.col("EegMastrNummer").cast("string").alias("eeg_mastr_nummer"),
            F.lit(technologie).alias("technologie"),
            F.col("Bundesland").cast("string").alias("bundesland"),
            F.col("Landkreis").cast("string").alias("landkreis"),
            F.col("Gemeinde").cast("string").alias("gemeinde"),
            F.col("Gemeindeschluessel").cast("string").alias("gemeindeschluessel"),
            F.col("Postleitzahl").cast("string").alias("plz"),
            F.col("Breitengrad").cast("double").alias("breitengrad"),
            F.col("Laengengrad").cast("double").alias("laengengrad"),
            F.col("Bruttoleistung").cast("double").alias("brutto_leistung_kw"),
            F.col("Nettonennleistung").cast("double").alias("netto_nennleistung_kw"),
            F.col("Inbetriebnahmedatum").cast("date").alias("inbetriebnahme_datum"),
            F.col("EegInbetriebnahmedatum").cast("date").alias("eeg_inbetriebnahme_datum"),
            F.col("EinheitBetriebsstatus").cast("string").alias("betriebs_status"),
            F.col("Anlagenbetreiber").cast("string").alias("anlagenbetreiber"),
        )
    )

# DEBUG: hydro only for fast iteration. Re-add the others before final run.
DEBUG_HYDRO_ONLY = False

# Build the unioned dimension
dim = (
    select_common("solar", "solar")
    .unionByName(select_common("wind", "wind"))
    .unionByName(select_common("biomass", "biomass"))
    .unionByName(select_common("hydro", "hydro"))
)

if DEBUG_HYDRO_ONLY:
    dim = select_common("hydro", "hydro")
else:
    dim = (
        select_common("solar", "solar")
        .unionByName(select_common("wind", "wind"))
        .unionByName(select_common("biomass", "biomass"))
        .unionByName(select_common("hydro", "hydro"))
    )

# Filter out rows we can't geo-locate
dim_clean = (
    dim
    .filter(F.col("mastr_nummer").isNotNull())
    .filter(F.col("plz").rlike(r"^\d{5}$"))
    .filter(F.col("bundesland").isNotNull())
)

# Materialize as a Delta table (overwrite — this is a full rebuild from bronze)
(
    dim_clean.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("eeg_dev.silver.dim_anlage")
)

# Quick stats
total = dim.count()
clean = dim_clean.count()
dropped = total - clean
print(f"Total bronze rows:    {total:>10,}")
print(f"Kept in dim_anlage:   {clean:>10,}")
print(f"Dropped (filters):    {dropped:>10,} ({100*dropped/total:.2f}%)")